# SK-2-Functions : Function Calling, Memory et Fonctionnalites Avancees

**Navigation** : [Index](README.md) | [<< 01-Intro](01-SemanticKernel-Intro.ipynb) | [03-Agents >>](03-SemanticKernel-Agents.ipynb)

***

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Utiliser **Function Calling** avec `FunctionChoiceBehavior` pour orchestrer automatiquement des fonctions
2. Configurer la **memoire vectorielle** avec l'API moderne (InMemoryStore, embeddings)
3. Integrer des modèles **Hugging Face** comme alternative a OpenAI
4. Implementer un **Groundedness Checking** pour reduire les hallucinations
5. Generer **plusieurs reponses** en un seul appel API

### Prerequis

- Python 3.10+
- Notebook [01-SemanticKernel-Intro](01-SemanticKernel-Intro.ipynb) complete
- Cle API OpenAI configuree dans `.env`

### Duree estimee : 50 minutes

***

## Sommaire

| Section | Contenu | Concepts cles |
|---------|---------|---------------|
| 1 | Installation et configuration | Kernel, services LLM |
| 2 | Chat avec KernelArguments | Templates, historique |
| 3 | **Function Calling moderne** | `FunctionChoiceBehavior.Auto()` |
| 4 | Memoire et Embeddings | InMemoryStore, Vector Search |
| 5 | Hugging Face Integration | Modèles open-source |
| 6 | Groundedness Checking | Anti-hallucination |
| 7 | Multi-Result | Generations multiples |

> **Note importante** : Les anciens `SequentialPlanner` et `FunctionCallingStepwisePlanner` sont **deprecies** depuis SK 1.30. Ce notebook utilise l'approche moderne avec `FunctionChoiceBehavior`.

In [1]:
# ============================
# Cellule : Installation & Imports
# ============================

# N'installez qu'une seule fois si nécessaire

# Imports de base
import os
import sys
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.functions import KernelArguments
from semantic_kernel.contents import ChatHistory

print("Imports et installation OK.")


Imports et installation OK.


## 1. Installation et configuration initiale

Cette première cellule effectue deux opérations essentielles :
1. **Installation de semantic-kernel** : Le SDK Python pour orchestrer des modèles de langage
2. **Imports de base** : Les modules fondamentaux pour interagir avec le kernel et gerer l'environnement

Le fichier `.env` doit contenir vos cles API (voir la section suivante pour le format attendu).

### Configuration du Kernel et du `.env`

Pour exécuter les exemples, un fichier `.env` dans le dossier du notebook doit comporter vos clés d'API. Exemple :

```plaintext
GLOBAL_LLM_SERVICE="OpenAI"        # ou AzureOpenAI, HuggingFace
OPENAI_API_KEY="sk-..."
OPENAI_CHAT_MODEL_ID="gpt-5-mini"
```

Le code lira `GLOBAL_LLM_SERVICE` pour décider quel connecteur instancier (OpenAI, Azure OpenAI ou Hugging Face) — voir la cellule d'initialisation ci-dessous.

## 2. Initialisation du Kernel avec service LLM dynamique

Cette cellule configure le Kernel en fonction du service LLM specifie dans le fichier `.env`. Le code supporte trois backends :
- **OpenAI** : API officielle d'OpenAI (GPT-4, GPT-3.5, etc.)
- **Azure OpenAI** : Deploiement Azure avec endpoint personnalise
- **Hugging Face** : Modèles open-source en local ou via le Hub

Le pattern utilise ici est courant : on lit la variable `GLOBAL_LLM_SERVICE` pour determiner dynamiquement quel connecteur instancier.

In [2]:
# ============================
# Cellule : Initialisation du Kernel
# ============================

from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatCompletion,
    AzureChatCompletion
)

# On charge le .env
load_dotenv()
global_llm_service = os.getenv("GLOBAL_LLM_SERVICE", "AzureOpenAI")

# Initialisation du Kernel
kernel = Kernel()

service_id = "default"

if global_llm_service.lower() == "openai":
    # Ajout du service OpenAI
    kernel.add_service(
        OpenAIChatCompletion(service_id=service_id),
    )
    print("Service OpenAI configuré.")
elif global_llm_service.lower() == "huggingface":
    # Ajout du service HuggingFace
    from semantic_kernel.connectors.ai.hugging_face import HuggingFaceTextCompletion
    kernel.add_service(
        HuggingFaceTextCompletion(
            service_id=service_id,
            ai_model_id="distilgpt2", # ex. pour text-generation
            task="text-generation"
        ),
    )
    print("Service Hugging Face configuré.")
else:
    # Par défaut : Azure OpenAI
    kernel.add_service(
        AzureChatCompletion(service_id=service_id),
    )
    print("Service Azure OpenAI configuré.")

Service OpenAI configuré.


## Choix du service LLM

Le code ci-dessus illustre un pattern important : **la configuration dynamique du service LLM**.

Selon la valeur de `GLOBAL_LLM_SERVICE` dans votre fichier `.env`, le Kernel se connectera a :
- **OpenAI** : API officielle avec `OpenAIChatCompletion`
- **Azure OpenAI** : Deploiement Azure avec `AzureChatCompletion`
- **Hugging Face** : Modèles open-source avec `HuggingFaceTextCompletion`

Cette flexibilite permet de switcher facilement entre fournisseurs sans modifier le code applicatif.

### Chat Basique avec KernelArguments

L'idée : on crée une fonction de chat qui prend :
- l’historique de conversation (objet `ChatHistory`)
- un `user_input`
et on stocke le tout dans un `KernelArguments`.  

Cela permet d'alimenter un prompt (via un template) qui contient la variable `history` et `user_input`.

In [3]:
# ============================
# Cellule : Extrait d'un Chat Minimal
# ============================

# Exemple de prompt
chat_prompt = """
{{$history}}
User: {{$user_input}}
ChatBot:
"""

# Création d'une fonction sémantique "chat"
from semantic_kernel.prompt_template import PromptTemplateConfig
from semantic_kernel.prompt_template.input_variable import InputVariable

pt_config = PromptTemplateConfig(
    template=chat_prompt,
    name="chatFunction",
    template_format="semantic-kernel",
    input_variables=[
        InputVariable(name="user_input", description="User's message"),
        InputVariable(name="history", description="Conversation history"),
    ]
)

chat_function = kernel.add_function(
    function_name="chat",
    plugin_name="myChatPlugin",
    prompt_template_config=pt_config
)

chat_history = ChatHistory()
chat_history.add_system_message("Vous êtes un chatbot utile spécialisé en recommandations de livres.")

async def chat_kernel(input_text: str):
    print(f"[Utilisateur] : {input_text}")
    response = await kernel.invoke(
        chat_function,
        KernelArguments(user_input=input_text, history=str(chat_history))
    )
    print(f"[ChatBot] : {response}")
    # Mise à jour de l'historique
    chat_history.add_user_message(input_text)
    chat_history.add_assistant_message(str(response))


# Test
await chat_kernel("Salut, peux-tu me conseiller un livre sur la philosophie antique ?")
await chat_kernel("Merci, tu peux détailler un peu plus la période concernée ?")

[Utilisateur] : Salut, peux-tu me conseiller un livre sur la philosophie antique ?


[ChatBot] : Salut — oui bien sûr. Voici plusieurs suggestions selon ce que tu cherches (introduction générale, textes originaux, ou approche pratique) :

1. Pierre Hadot — Qu'est-ce que la philosophie antique ?  
   - Idéal si tu veux comprendre la philosophie antique comme « manière de vivre » plutôt que simple théorie. Clair, profond et très recommandé comme introduction.

2. Platon — un dialogue (par ex. La République ou Apologie de Socrate)  
   - Pour lire la source. La République aborde la justice, la politique et la théorie des idées ; l'Apologie est un bon point d'entrée sur Socrate.

3. Aristote — Éthique à Nicomaque  
   - Fondamental si tu t'intéresses à la morale pratique et à la conception antique de la vertu. Très influent et accessible par morceaux.

4. Stoïcisme pratique : Marc Aurèle — Pensées pour moi-même (Méditations) ou Épictète — Manuel (Enchiridion)  
   - Textes brefs, concrets et utilisables au quotidien. Bon choix si tu veux de l’éthique pragmatique.

5. Sénèq

[ChatBot] : Bonne question — voici un panorama clair de ce que recouvre habituellement la « philosophie antique » (avec dates approximatives, acteurs et thèmes principaux).

Périodisation et figures clés
- Présocratiques (fin du VIe — Ve siècle av. J.-C.)  
  - Thèmes : origines du cosmos, principe (archê), nature.  
  - Figures : Thalès, Anaximandre, Héraclite, Parménide, Démocrite (atomisme).  

- Période classique (Ve — IVe siècle av. J.-C.)  
  - Thèmes : éthique, politique, connaissance, ontologie.  
  - Figures : Socrate (par les dialogues de Platon), Platon (dialogues), Aristote (logique, métaphysique, éthique).

- Période hellénistique (début IIIe — Ier siècle av. J.-C.)  
  - Thèmes : philosophie comme art de vivre, éthique pratique, réponses à l’incertitude politique.  
  - Écoles : stoïcisme (Zénon, Chrysippe), épicurisme (Épicure), scepticisme (Pyrrhon, Académie sceptique).

- Période romaine et impériale (Ier siècle av. J.-C. — IIIe siècle ap. J.-C.)  
  - Thèmes : adaptat

### Interprétation : Chat Conversationnel avec Historique

**Sortie obtenue** : Un chatbot conversationnel qui maintient le contexte entre les échanges.

| Échange | Utilisateur | Assistant | Contexte maintenu |
|---------|-------------|-----------|-------------------|
| **1** | "Salut, peux-tu me conseiller un livre sur la philosophie antique ?" | Demande une précision (panorama / textes traduits / courant précis) **et** propose déjà une sélection dans le même tour | Message système (spécialisation livres) |
| **2** | "Merci, tu peux détailler un peu plus la période concernée ?" | Déroule les phases de la période (présocratiques → classique → hellénistique → romaine impériale → néoplatonisme) | Se rappelle du sujet (philosophie antique) |

**Points clés** :

1. **Architecture du chat avec historique** :
   ```python
   ChatHistory → KernelArguments → PromptTemplate → LLM → Update ChatHistory
        ↓              ↓                 ↓            ↓              ↓
   Contexte      Variables        Prompt final   Réponse    Nouveau contexte
   ```

2. **Composants essentiels** :

   | Composant | Rôle | Exemple |
   |-----------|------|---------|
   | `ChatHistory` | Stocke tous les messages (système, utilisateur, assistant) | Conserve le contexte |
   | `PromptTemplateConfig` | Définit le template avec variables `{{$history}}`, `{{$user_input}}` | Structure du prompt |
   | `KernelArguments` | Passe les valeurs dynamiques au template | `user_input`, `history` |
   | `chat_history.add_*_message()` | Met à jour l'historique après chaque échange | Persistence mémoire |

3. **Gestion du message système** :
   ```python
   chat_history.add_system_message("Vous êtes un chatbot utile spécialisé en recommandations de livres.")
   ```
   - Définit le comportement et la personnalité du bot
   - Reste présent dans tout l'historique
   - Peut spécifier le format de réponse, le ton, les contraintes

4. **Exemple de prompt généré (2ème échange)** :
   ```
   [SYSTEM] Vous êtes un chatbot utile spécialisé en recommandations de livres.
   [USER] Salut, peux-tu me conseiller un livre sur la philosophie antique ?
   [ASSISTANT] Propose une sélection (Hadot, Platon, Aristote, Stoïciens...) et demande une précision
   [USER] Merci, tu peux détailler un peu plus la période concernée ?
   [CHATBOT] <à générer>
   ```

**Limitations et améliorations** :

1. **Limite de contexte** :
   - Le `ChatHistory` peut devenir très long et dépasser le context window du LLM
   - Solutions :
     - Tronquer l'historique (garder N derniers messages)
     - Résumer l'historique (summarization)
     - Mémoire vectorielle (stocker l'historique dans un vector store)

2. **Code amélioré avec gestion de contexte** :
   ```python
   MAX_HISTORY_LENGTH = 10  # Garder les 10 derniers échanges
   
   if len(chat_history.messages) > MAX_HISTORY_LENGTH:
       # Garder le message système + N derniers messages
       system_msg = chat_history.messages[0]
       recent_msgs = chat_history.messages[-(MAX_HISTORY_LENGTH-1):]
       chat_history = ChatHistory()
       chat_history.add_message(system_msg)
       for msg in recent_msgs:
           chat_history.add_message(msg)
   ```

**Comparaison avec d'autres patterns** :

| Pattern | Mémoire | Use case |
|---------|---------|----------|
| **Stateless** (prompt fixe) | Aucune | FAQ simple |
| **ChatHistory** (ce notebook) | Court terme (session) | Chatbot conversationnel |
| **Vector Memory** (notebook 05) | Long terme (permanent) | RAG, knowledge base |
| **Agents** (notebook 03) | Multi-agents avec état partagé | Workflows complexes |

**Cas d'usage typiques** :
- **Support client** : Résoudre un problème en plusieurs étapes
- **Assistant personnel** : Planification, organisation avec continuité
- **Tuteur éducatif** : Adaptation progressive au niveau de l'apprenant

### Exercice 1 : Chatbot avec gestion de contexte avancee

**Objectif** : Implementer un chatbot qui limite automatiquement la taille de l'historique de conversation pour eviter de depasser la fenêtre de contexte du LLM.

Le chat basique avec `ChatHistory` accumule tous les messages indefiniment. En production, il faut tronquer l'historique quand il devient trop long, tout en preservant le message système.

**Indices** :
- `# Étape 1` : Créer une fonction `trim_history(history, max_messages)` qui garde le message système + N derniers echanges
- `# Étape 2` : Compter les messages et verifier si `len(history.messages) > max_messages`
- `# Étape 3` : Reconstruire un nouvel historique avec le system message + les messages recents
- `# Indice` : Le message système est toujours `history.messages[0]` (ou filtrer par rôle `AuthorRole.SYSTEM`)

In [4]:
from semantic_kernel.contents import ChatHistory

def trim_history(history: ChatHistory, max_messages: int = 10) -> ChatHistory:
    """
    Tronque l'historique pour garder le message systeme + les N derniers messages.
    
    # Etape 1 : Verifier si l'historique depasse max_messages
    # Etape 2 : Separer le message systeme (premier message) des messages de conversation
    # Etape 3 : Reconstruire un nouvel historique avec system + derniers messages
    
    # Indice : ChatHistory() cree un nouvel historique vide
    # Indice : history.add_message(msg) ajoute un message existant au nouvel historique
    
    Args:
        history: L'historique de conversation original
        max_messages: Nombre maximum de messages a conserver (hors system)
    
    Returns:
        ChatHistory tronque si necessaire, sinon l'original
    """
    # TODO etudiant : implementer la troncature d'historique
    return history  # TODO etudiant : retourner l'historique tronque

# Test avec un historique long
test_history = ChatHistory()
test_history.add_system_message("Tu es un assistant utile.")
for i in range(15):
    test_history.add_user_message(f"Question {i+1}")
    test_history.add_assistant_message(f"Reponse {i+1}")

print(f"Avant trim : {len(test_history.messages)} messages")
trimmed = trim_history(test_history, max_messages=6)
print(f"Apres trim : {len(trimmed.messages)} messages")
print(f"Premier message conserve (system) : {trimmed.messages[0].content[:30]}...")
print(f"Dernier message : {trimmed.messages[-1].content}")

Avant trim : 31 messages
Apres trim : 31 messages
Premier message conserve (system) : Tu es un assistant utile....
Dernier message : Reponse 15


## 3. Function Calling Moderne

### Evolution des Planners vers Function Calling

Les **Planners** (SequentialPlanner, FunctionCallingStepwisePlanner) ont ete **deprecies** dans Semantic Kernel 1.30+. Microsoft recommande maintenant d'utiliser :

| Ancienne approche | Nouvelle approche | Avantages |
|-------------------|-------------------|-----------|
| `SequentialPlanner` | `FunctionChoiceBehavior.Auto()` | Moins de tokens, plus fiable |
| `FunctionCallingStepwisePlanner` | Agent avec plugins | Pattern ReAct natif |
| Planification XML | Function calling OpenAI | Standard API |

### Qu'est-ce que Function Calling ?

**Function Calling** permet au LLM de :
1. Analyser la requête utilisateur
2. Decider automatiquement quelle(s) fonction(s) appeler
3. Extraire les paramètres depuis le contexte
4. Retourner le résultat au LLM pour formulation finale

C'est le pattern **ReAct** (Reasoning + Acting) integre directement dans l'API OpenAI.

### Configuration de FunctionChoiceBehavior

Semantic Kernel propose plusieurs modes de Function Calling :

```python
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

# Mode Auto : le LLM decide quand appeler les fonctions
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

# Mode Required : force l'appel d'au moins une fonction
settings.function_choice_behavior = FunctionChoiceBehavior.Required()

# Mode None : desactive le function calling
settings.function_choice_behavior = FunctionChoiceBehavior.NoneInvoke()
```

### Paramètres avances

| Paramètre | Description | Valeur par defaut |
|-----------|-------------|-------------------|
| `auto_invoke` | Executer automatiquement les fonctions | `True` |
| `filters` | Filtrer les fonctions disponibles | `None` |
| `maximum_auto_invoke_attempts` | Limite d'appels | `5` |

L'exemple suivant montre comment configurer un kernel avec plusieurs plugins et laisser le LLM orchestrer automatiquement.

### API de memoire vectorielle mise a jour

**Note de version** : L'API de memoire a evolue dans Semantic Kernel. Les anciennes classes comme `VolatileMemoryStore` et `SemanticTextMemory` sont remplacees par :
- `InMemoryStore` : Stockage en memoire volatile
- Services d'embedding dedies avec `OpenAITextEmbedding`

Cette cellule montre le pattern moderne pour configurer la memoire sémantique.

In [5]:
# ============================
# Function Calling Moderne avec FunctionChoiceBehavior
# ============================

from semantic_kernel.core_plugins.text_plugin import TextPlugin
from semantic_kernel.core_plugins.math_plugin import MathPlugin
from semantic_kernel.functions import KernelFunctionFromPrompt, kernel_function
from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

# Charger des plugins avec des fonctions utiles
plugins_directory = "./prompt_template_samples/"
writer_plugin = kernel.add_plugin(plugin_name="WriterPlugin", parent_directory=plugins_directory)
text_plugin = kernel.add_plugin(plugin=TextPlugin(), plugin_name="TextPlugin")
math_plugin = kernel.add_plugin(plugin=MathPlugin(), plugin_name="MathPlugin")

# Creer une fonction semantique pour la poesie
shakespeare_func = KernelFunctionFromPrompt(
    function_name="Shakespeare",
    plugin_name="WriterPlugin",
    prompt="""
{{$input}}

Rewrite the above in the style of Shakespeare.
""",
    prompt_execution_settings=OpenAIChatPromptExecutionSettings(
        service_id=service_id,
        # temperature not set: gpt-5-mini only supports default value (1)
    ),
    description="Rewrite the input in the style of Shakespeare.",
)
kernel.add_function(plugin_name="WriterPlugin", function=shakespeare_func)

# Lister les fonctions disponibles
print("Fonctions disponibles pour le LLM :")
print("-" * 50)
for plugin_name, plugin in kernel.plugins.items():
    for function_name, function in plugin.functions.items():
        desc = function.description[:50] + "..." if function.description and len(function.description) > 50 else function.description
        print(f"  {plugin_name}.{function_name}: {desc}")

# ============================
# Exemple : Function Calling Auto
# ============================
print("\n" + "=" * 50)
print("DEMO : Function Calling avec FunctionChoiceBehavior.Auto()")
print("=" * 50)

# Configurer les settings avec function calling auto
execution_settings = OpenAIChatPromptExecutionSettings(
    service_id=service_id,
    # temperature not set: gpt-5-mini only supports default value (1)
    function_choice_behavior=FunctionChoiceBehavior.Auto(
        auto_invoke=True,  # Execute automatiquement les fonctions
        filters={"included_plugins": ["TextPlugin", "MathPlugin"]}  # Limiter aux plugins
    )
)

# Prompt qui necessite l'appel de fonctions
user_request = """
J'ai besoin d'aide pour deux choses :
1. Convertir le texte "hello world" en majuscules
2. Calculer 25 + 17
"""

print(f"\nRequete utilisateur : {user_request}")

# Creer une fonction de chat avec function calling
from semantic_kernel.contents import ChatHistory

chat_history = ChatHistory()
chat_history.add_system_message("Tu es un assistant qui utilise les outils disponibles pour aider l'utilisateur.")
chat_history.add_user_message(user_request)

# Invoquer avec function calling
chat_service = kernel.get_service(service_id)
result = await chat_service.get_chat_message_content(
    chat_history=chat_history,
    settings=execution_settings,
    kernel=kernel
)

print(f"\nReponse du LLM (avec function calling) :")
print(result.content)

Fonctions disponibles pour le LLM :
--------------------------------------------------
  myChatPlugin.chat: 
  WriterPlugin.Acronym: Generate an acronym for the given concept or phras...
  WriterPlugin.AcronymGenerator: Given a request to generate an acronym from a stri...
  WriterPlugin.AcronymReverse: Given a single word or acronym, generate the expan...
  WriterPlugin.Brainstorm: Given a goal or topic description generate a list ...
  WriterPlugin.EmailGen: Write an email from the given bullet points
  WriterPlugin.EmailTo: Turn bullet points into an email to someone, using...
  WriterPlugin.EnglishImprover: Translate text to English and improve it
  WriterPlugin.NovelChapter: Write a chapter of a novel.
  WriterPlugin.NovelChapterWithNotes: Write a chapter of a novel using notes about the c...
  WriterPlugin.NovelOutline: Generate a list of chapter synopsis for a novel or...
  WriterPlugin.Rewrite: Automatically generate compact notes for any text ...
  WriterPlugin.ShortPoem: Turn


Reponse du LLM (avec function calling) :
Voici les résultats :

1. "hello world" en majuscules -> HELLO WORLD
2. 25 + 17 = 42


### Interprétation : Architecture Function Calling

**Sortie obtenue** : Le LLM a automatiquement identifié et exécuté deux fonctions sans code manuel :
- `TextPlugin.uppercase("hello world")` → "HELLO WORLD"
- `MathPlugin.Add(25, 17)` → 42

| Aspect | Détail | Signification |
|--------|--------|---------------|
| **Plugins chargés** | WriterPlugin (17 fonctions), TextPlugin (5), MathPlugin (2) | Bibliothèque d'outils disponibles pour le LLM |
| **Mode Auto** | `FunctionChoiceBehavior.Auto()` | Le LLM décide automatiquement quand et comment appeler les fonctions |
| **Filtres** | `included_plugins: ["TextPlugin", "MathPlugin"]` | Limitation des fonctions accessibles pour sécurité/performance |
| **Auto-invoke** | `auto_invoke=True` | Exécution automatique sans intervention manuelle |

**Points clés** :

1. **Pattern ReAct intégré** : Le LLM raisonne (analyse la requête) puis agit (appelle les fonctions)
2. **Chaînage transparent** : Le LLM peut enchaîner plusieurs appels de fonctions si nécessaire
3. **Extraction de paramètres** : Les valeurs sont extraites du contexte naturel ("25 + 17" → `Add(25, 17)`)
4. **Formulation finale** : Le résultat brut des fonctions est reformulé dans une réponse naturelle

**Note technique** : Cette approche remplace les anciens Planners (SequentialPlanner, FunctionCallingStepwisePlanner) qui généraient du XML intermédiaire. Function Calling utilise directement l'API OpenAI, ce qui réduit les tokens et améliore la fiabilité.

**Comparaison avec Agents** :
- **Function Calling** : Orchestration simple, un LLM qui appelle des fonctions
- **Agents** (voir notebook 03) : Plusieurs agents qui collaborent, chacun avec ses propres plugins

**Cas d'usage typiques** :
- Assistants conversationnels avec accès à des APIs
- Chatbots d'entreprise avec fonctions métier (recherche produits, calculs, requêtes DB)
- Outils d'aide à la décision avec fonctions analytiques

### Exercice 2 : Plugin de conversion d'unites avec Function Calling

**Objectif** : Créer un plugin `UnitConverterPlugin` avec des fonctions de conversion (temperature, distance) et le rendre disponible via Function Calling automatique.

Le Function Calling automatique est puissant quand les plugins offrent des fonctions bien decrites avec des types précis. Vous allez créer un plugin que le LLM pourra appeler automatiquement pour convertir des unites dans une conversation.

**Indices** :
- `# Étape 1` : Créer deux fonctions `celsius_to_fahrenheit` et `km_to_miles` avec `@kernel_function`
- `# Étape 2` : Ajouter des descriptions claires et des annotations de type pour les paramètres
- `# Étape 3` : Tester avec une requête naturelle comme "Combien font 37 degrés Celsius en Fahrenheit ?"
- `# Indice` : `F = C * 9/5 + 32` et `miles = km * 0.621371`

In [6]:
# Annotated est requis par les signatures des kernel_functions ci-dessous.
from typing import Annotated

class UnitConverterPlugin:
    """
    Plugin de conversion d'unites pour Function Calling.
    
    # Etape 1 : Implementer celsius_to_fahrenheit avec @kernel_function
    # Etape 2 : Implementer km_to_miles avec @kernel_function
    # Etape 3 : Ajouter des descriptions precises pour que le LLM sache quand les utiliser
    """
    
    @kernel_function(description="Convertit une temperature de Celsius en Fahrenheit")
    def celsius_to_fahrenheit(self, celsius: Annotated[float, "Temperature en degres Celsius"]) -> str:
        """
        # Indice : Appliquer la formule F = C * 9/5 + 32
        # Indice : Retourner une string descriptive comme "37.0 C = 98.6 F"
        """
        # TODO etudiant : implementer la conversion
        return "Exercice a completer"  # TODO etudiant : retourner le resultat formatte
    
    @kernel_function(description="Convertit une distance de kilometres en miles")
    def km_to_miles(self, km: Annotated[float, "Distance en kilometres"]) -> str:
        """
        # Indice : Appliquer la formule miles = km * 0.621371
        """
        # TODO etudiant : implementer la conversion
        return "Exercice a completer"  # TODO etudiant : retourner le resultat formatte

# Test du plugin seul
converter = UnitConverterPlugin()
print(converter.celsius_to_fahrenheit(37.0))  # Attendu : "37.0 C = 98.6 F"
print(converter.km_to_miles(100.0))  # Attendu : "100.0 km = 62.1 miles"

# TODO etudiant : ajouter le plugin au kernel et tester avec Function Calling Auto
# kernel.add_plugin(UnitConverterPlugin(), plugin_name="converter")
# Puis invoquer avec une requete naturelle via le chat service
print("Exercice a completer : integrez le plugin dans le kernel avec Function Calling")

Exercice a completer
Exercice a completer
Exercice a completer : integrez le plugin dans le kernel avec Function Calling


### Mémoire & Embeddings

**SemanticTextMemory** permet de stocker des textes (avec un embedding) dans un store :
- `VolatileMemoryStore` (en mémoire)
- ou connecteurs vers Pinecone, Azure Cognitive Search, Qdrant, etc.

On peut ensuite effectuer des requêtes sémantiques :  
`await memory.search("MaCollection", "Quelle est mon budget pour 2024 ?")`

### Verification du GroundingPlugin

Le plugin de Grounding (ancrage) permet de :
1. **Extraire les entites** d'un texte resume
2. **Verifier** chaque entite par rapport a un texte source de reference
3. **Corriger** le resume en supprimant les informations non fondees

Ce mécanisme est essentiel pour reduire les hallucinations des LLMs et garantir que les reponses sont ancrees dans des faits verifiables.

In [7]:
# ============================
# Cellule : Extrait Memoire
# ============================

# Memoire vectorielle REELLE : l'embedder maison de la flotte (Qwen3-Embedding-4B,
# servi par vLLM derriere une API compatible OpenAI) produit les vecteurs -- le calcul
# se fait sur le GPU de la flotte, sans fournisseur externe. URL de base, cle et nom
# du modele viennent de l'environnement, sans valeur par defaut litterale
# (secrets-hygiene.md). Aucun repli silencieux vers un autre service : changer de
# modele change la dimension des vecteurs et invaliderait les lectures chiffrees.
import os

import numpy as np
from openai import AsyncOpenAI
from semantic_kernel.connectors.ai.open_ai import OpenAITextEmbedding


def _variable_requise(nom: str) -> str:
    """Lit une variable d'environnement et echoue explicitement si elle manque."""
    valeur = os.environ.get(nom)
    if not valeur:
        raise RuntimeError(f"{nom} absent de l'environnement (voir .env.example)")
    return valeur


EMBEDDING_BASE_URL = _variable_requise("EMBEDDING_BASE_URL")
EMBEDDING_API_KEY = _variable_requise("EMBEDDING_API_KEY")
EMBEDDING_MODEL = _variable_requise("EMBEDDING_MODEL")

client = AsyncOpenAI(base_url=EMBEDDING_BASE_URL, api_key=EMBEDDING_API_KEY)
embedding_service = OpenAITextEmbedding(
    service_id="embeddingService",
    ai_model_id=EMBEDDING_MODEL,
    async_client=client,
)

# Contenu de la memoire : quelques faits personnels
memoire = {
    "budget_2024": "Le budget 2024 est de 100 kEUR.",
    "budget_2023": "Le budget 2023 etait de 70 kEUR.",
    "conges": "Il reste 12 jours de conges en 2024.",
    "voiture": "La voiture de service est une Megane electrique.",
}
textes = list(memoire.values())

# Embeddings REELS du corpus et de la requete (generate_embeddings rend un ndarray)
vecteurs = np.asarray(await embedding_service.generate_embeddings(textes))
requete = "Quel est mon budget pour 2024 ?"
vecteur_requete = np.asarray(await embedding_service.generate_embeddings([requete]))[0]
# Dimension mesuree sur le premier embedding : elle sert de source unique a la
# lecture chiffree ci-dessous, jamais une constante recopiee en dur.
DIM = vecteurs.shape[1]
print(f"Embedder : {EMBEDDING_MODEL} (embedder maison de la flotte), {DIM} dimensions, {len(textes)} entrees")

# Recherche semantique : similarite cosinus entre la requete et chaque entree memoire
scores = (vecteurs @ vecteur_requete) / (
    np.linalg.norm(vecteurs, axis=1) * np.linalg.norm(vecteur_requete)
)
classement = sorted(zip(scores, memoire, textes), key=lambda couple: -couple[0])
for rang, (score, cle, texte) in enumerate(classement, 1):
    print(f"  {rang}. cosinus = {score:.4f}  [{cle}]  {texte}")

print(f"\nReponse retenue (meilleur cosinus) : {classement[0][2]}")

# Contre-epreuve : une paraphrase qui n'emploie ni "budget" ni "2024"
paraphrase = "De combien d'argent je dispose cette annee ?"
vecteur_paraphrase = np.asarray(await embedding_service.generate_embeddings([paraphrase]))[0]
scores_paraphrase = (vecteurs @ vecteur_paraphrase) / (
    np.linalg.norm(vecteurs, axis=1) * np.linalg.norm(vecteur_paraphrase)
)
print(f"\nContre-epreuve -- paraphrase sans les mots-cles : {paraphrase!r}")
for rang, (score, cle) in enumerate(
    sorted(zip(scores_paraphrase, memoire), key=lambda couple: -couple[0]), 1
):
    print(f"  {rang}. cosinus = {score:.4f}  [{cle}]")


Embedder : qwen3-4b-awq-embedding (embedder maison de la flotte), 2560 dimensions, 4 entrees
  1. cosinus = 0.8269  [budget_2024]  Le budget 2024 est de 100 kEUR.
  2. cosinus = 0.6262  [budget_2023]  Le budget 2023 etait de 70 kEUR.
  3. cosinus = 0.5166  [conges]  Il reste 12 jours de conges en 2024.
  4. cosinus = 0.2346  [voiture]  La voiture de service est une Megane electrique.

Reponse retenue (meilleur cosinus) : Le budget 2024 est de 100 kEUR.

Contre-epreuve -- paraphrase sans les mots-cles : "De combien d'argent je dispose cette annee ?"
  1. cosinus = 0.4848  [budget_2024]
  2. cosinus = 0.4519  [budget_2023]
  3. cosinus = 0.4296  [conges]
  4. cosinus = 0.2315  [voiture]


### Interprétation : Mémoire Vectorielle et Embeddings

**Sortie obtenue** : une recherche sémantique **réelle** — l'embedder maison de la flotte (Qwen3-Embedding-4B) a vectorisé le corpus et la requête, et le classement par cosinus est **mesuré**, pas simulé.

| Composant | Technologie | Rôle |
|-----------|-------------|------|
| **Embedding Service** | `qwen3-4b-awq-embedding` (Qwen3-Embedding-4B), servi par la flotte derrière une API compatible OpenAI | Convertit le texte en vecteurs denses (2560 dimensions) |
| **Recherche** | Similarité cosinus (numpy) | Classe les entrées de la mémoire par proximité sémantique |
| **Query** | "Quel est mon budget pour 2024 ?" | Requête posée à l'embedder, comme n'importe quel texte |
| **Résultat** | `budget_2024` en tête, devant `budget_2023` | Scores de cosinus mesurés dans la cellule ci-dessus |

**Points clés** :

1. **Ce que cette mesure prouve, et ce qu'elle ne prouve pas.**
   Le classement est réel : quatre cosinus calculés sur quatre vecteurs réels, `budget_2024` (0.8269) devant `budget_2023` (0.6262). Mais la requête contient littéralement les mots « budget » et « 2024 » : ce classement ne distingue donc pas une recherche sémantique d'une recherche par mots-clés.
   La contre-épreuve imprimée par la cellule ci-dessus répond à cette objection : une paraphrase qui évite ces deux mots (« De combien d'argent je dispose cette annee ? ») maintient `budget_2024` en tête (0.4848) devant `budget_2023` (0.4519). L'écart (0.0329) est plus net que l'écart entre les deux entrées ne le laisserait craindre : l'embedder garde la trace de l'année dans des phrases qui ne la nomment pas.
   Cette robustesse est une propriété du modèle, pas une garantie : un embedder moins discriminant peut inverser ce classement sur le même corpus. C'est mesurable ici en changeant `EMBEDDING_MODEL` — et c'est la raison pour laquelle le carnet interdit tout repli silencieux d'un embedder vers un autre.

2. **Évolution de l'API SK** :
   ```python
   # Ancienne approche (dépréciée)
   memory_store = VolatileMemoryStore()
   memory = SemanticTextMemory(store, embedding)
   await memory.save_information("collection", "text", "id")

   # Nouvelle approche (2024+)
   store = InMemoryStore()
   embedding = OpenAITextEmbedding(ai_model_id="text-embedding-3-small")
   kernel.add_service(embedding)
   ```
   Dans la cellule ci-dessus, le même service `OpenAITextEmbedding` est pointé vers l'endpoint de l'embedder maison : l'API est identique, seuls changent le client — `base_url` et la clé, tous deux lus dans l'environnement — et le `ai_model_id`.

3. **Backends vectoriels supportés** :

   | Backend | Use case | Persistence |
   |---------|----------|-------------|
   | `InMemoryStore` | Dev, tests, démos | Volatile |
   | Qdrant | Production, scalabilité | Persistant |
   | Pinecone | Cloud, managed service | Persistant |
   | Azure AI Search | Intégration Azure | Persistant |
   | Chroma | Local, open-source | Persistant |

4. **Coût et performance** :
   - Ici : **aucun frais d'API externe par requête** — l'embedder tourne sur le GPU de la flotte, derrière une API compatible OpenAI ; l'électricité et le matériel ne sont pas comptés dans ce chiffre.
   - La bascule vers OpenAI (`text-embedding-3-small`, ~$0.00002 / 1K tokens) ne demande que trois valeurs : le `base_url` du client, la clé lue dans l'environnement (`api_key`) et le `ai_model_id`. Les deux services exigent une clé réelle : c'est précisément ce que garantit la lecture stricte par `_variable_requise`.
   - Latence : **non mesurée ici** — l'expérience porte sur quatre documents, et un balayage exact croît linéairement avec le nombre de vecteurs. Les « dizaines de ms sur 10K documents » qu'on lit couramment valent pour un index de backend vectoriel, pas pour ce carnet : rien dans les sorties ci-dessus ne les mesure.

**Note technique** : Le notebook [05-VectorStores](05-SemanticKernel-VectorStores.ipynb) couvre l'intégration complète avec Qdrant pour des systèmes RAG (Retrieval-Augmented Generation) en production — avec le même embedder maison.

**Cas d'usage typiques** :
- **FAQ intelligente** : Recherche dans une base de connaissances
- **RAG** : Enrichir les réponses LLM avec des documents métier
- **Chatbot avec mémoire** : Retrouver des conversations passées par similarité


### Evolution de l'API de memoire

L'API de memoire vectorielle de Semantic Kernel a evolue significativement :

**Ancienne approche (deprecie)** :
- `VolatileMemoryStore` + `SemanticTextMemory`
- Méthodes `.save_information()` et `.search()`

**Nouvelle approche (2024+)** :
- `InMemoryStore` pour le stockage volatile
- Services d'embedding dedies (`OpenAITextEmbedding`)
- Integration directe avec les connecteurs vectoriels (Pinecone, Qdrant, Azure AI Search)

Cette evolution permet une meilleure separation des responsabilites et une integration plus flexible avec divers backends.

### Configuration des reponses multiples

La fonctionnalite Multi-Result permet d'obtenir plusieurs completions pour un même prompt en un seul appel API. Cela est utile pour :
- **Generer des alternatives** : Obtenir plusieurs variations d'une reponse
- **Evaluer la diversite** : Comparer différentes formulations
- **Sélection humaine** : Presenter plusieurs options a l'utilisateur

Le paramètre `number_of_responses` dans les settings contrôle le nombre de reponses retournees.

### Hugging Face Intégration

Semantic Kernel peut se connecter à Hugging Face localement ou via API.  
Exemple :  
```python
from semantic_kernel.connectors.ai.hugging_face import HuggingFaceTextCompletion

hf_service = HuggingFaceTextCompletion(
    service_id="textHF", ai_model_id="distilgpt2", task="text-generation"
)
kernel.add_service(hf_service)
```

Ensuite, on peut enregistrer une fonction sémantique ou invoquer directement `hf_service.get_text_contents(...)`.

### Groundedness Checking

Pour éviter les “hallucinations” d’un résumé :  
1. Extrait la liste d'entités du résumé.  
2. Vérifie la correspondance de chaque entité avec le texte source (référence).  
3. Retire ou corrige les entités non-fondées.

Cela se fait via un plugin “GroundingPlugin” (ex. ExtraitEntities, ReferenceCheckEntities, ExciseEntities).

In [8]:
# ============================
# Cellule : Groundedness Checking
# ============================

# Suppose qu'on a un "grounding_text" = un texte source
grounding_text = """
Votre budget 2024 est de 100k euros.
Vous vivez à Genève.
Vous avez investi 50k en actions.
"""

# Suppose qu'on a un résumé "faux"
summary_text = """
Mon budget 2024 est de 200k euros.
J'habite à Milan.
"""

plugins_directory = "./prompt_template_samples/"

# On appelle un plugin (hypothétique) "GroundingPlugin" comportant 3 fonctions
try:
    grounding_plugin = kernel.add_plugin(parent_directory=plugins_directory, plugin_name="GroundingPlugin")
    extract_entities = grounding_plugin["ExtractEntities"]
    check_entities = grounding_plugin["ReferenceCheckEntities"]
    excise_entities = grounding_plugin["ExciseEntities"]
    
    # 1) Extraire entités avec les bonnes variables
    ext_result = await kernel.invoke(
        extract_entities, 
        KernelArguments(
            input=summary_text,
            topic="entities",  # Variable attendue par le template
            example_entities="Person, Location, Organization",  # Variable attendue
            allow_dangerously_set_content=True
        )
    )
    print("Entités détectées:", ext_result)
    
    # 2) Vérifier correspondance - Important: convertir ext_result en string
    ext_result_str = str(ext_result.value) if hasattr(ext_result, 'value') else str(ext_result)
    check_result = await kernel.invoke(
        check_entities, 
        KernelArguments(
            input=ext_result_str,  # Utiliser la version string
            reference_context=grounding_text,
            topic="entities",  # Ajouter les variables attendues
            allow_dangerously_set_content=True  # IMPORTANT: autoriser le contenu complexe
        )
    )
    print("Entités non-fondées:", check_result)
    
    # 3) Retirer entités non-fondées du summary
    check_result_str = str(check_result.value) if hasattr(check_result, 'value') else str(check_result)
    excision = await kernel.invoke(
        excise_entities, 
        KernelArguments(
            input=summary_text,
            ungrounded_entities=check_result_str,  # Utiliser la version string
            allow_dangerously_set_content=True
        )
    )
    print("Summary corrigé:", excision)
    
except Exception as e:
    print(f"Note: Le plugin GroundingPlugin n'est pas disponible ou fonctionnel: {e}")
    print("Démonstration alternative:")
    print(f"Texte source: {grounding_text[:50]}...")
    print(f"Résumé analysé: {summary_text[:50]}...")
    print("Entités potentiellement problématiques: Milan (au lieu de Genève), 200k€ (au lieu de 100k€)")

Entités détectées: - 2024 — Year (date) mentioned in the sentence "Mon budget 2024..."
- 200k euros — Monetary amount / currency value (budget amount) shown as "200k euros"
- Milan — Location (city) mentioned in "J'habite à Milan."

<entities>
- 2024
- 200k euros
- Milan
</entities>


Entités non-fondées: <ungrounded_entities>
- 200k euros
- Milan
</ungrounded_entities>


Summary corrigé: Mon budget 2024 est fixé.
J'habite ailleurs.


### Interprétation : Groundedness Checking (Vérification d'ancrage)

**Sortie obtenue** : Le système a détecté et corrigé les hallucinations dans un résumé en comparant avec un texte source.

| Étape | Résultat | Explication |
|-------|----------|-------------|
| **1. Extraction** | `<entities>` 2024, 200k euros, Milan `</entities>` | Entités factuelles identifiées dans le résumé |
| **2. Vérification** | `<ungrounded_entities>` 200k euros, Milan `</ungrounded_entities>` | Comparaison au texte source (100k ≠ 200k, Genève ≠ Milan) |
| **3. Correction** | Résumé corrigé imprimé par la cellule ci-dessus | Les deux entités non fondées en sont retirées : ni le montant ni la ville n'y survivent |

**Points clés** :

1. **Problème des hallucinations** :
   - Les LLMs peuvent générer des informations plausibles mais fausses
   - Particulièrement problématique en production (santé, finance, légal)
   - Le Groundedness Checking ancre les réponses dans des faits vérifiables

2. **Architecture du GroundingPlugin** :
   ```
   ExtractEntities → ReferenceCheckEntities → ExciseEntities
        ↓                      ↓                       ↓
   Liste entités      Entités invalides      Texte corrigé
   ```

3. **Entités détectées dans l'exemple** :

   | Entité résumé | Texte source | Statut |
   |---------------|--------------|--------|
   | 200k euros | 100k euros | ❌ Incohérence (supprimée) |
   | Milan | Genève | ❌ Hallucination (supprimée) |

4. **Limites de l'approche** :
   - Les deux entités incohérentes (200k euros vs 100k euros, Milan vs Genève) ont été retirées du résumé corrigé
   - Nécessite un prompt bien calibré pour détecter toutes les incohérences
   - Fonctionne mieux avec des entités discrètes (lieux, noms) qu'avec des valeurs numériques

**Note technique** : Le paramètre `allow_dangerously_set_content=True` est nécessaire pour passer du contenu structuré (XML) entre les fonctions. En production, il faut valider ce contenu pour éviter les injections.

**Améliorations possibles** :
```python
# Vérification multi-passes
for i in range(3):  # Itérer jusqu'à ce qu'il n'y ait plus d'entités non-fondées
    entities = extract_entities(summary)
    ungrounded = check_entities(entities, source)
    if not ungrounded:
        break
    summary = excise_entities(summary, ungrounded)
```

**Cas d'usage typiques** :
- **Résumés automatiques** : Garantir que le résumé ne contient que des infos du document source
- **Chatbots réglementés** : Assurer la conformité des réponses aux documents officiels
- **Fact-checking** : Valider automatiquement les affirmations d'un texte

### Résultats du Groundedness Checking

Le processus de verification d'ancrage comprend trois étapes :

1. **Extraction** : Identification des entites factuelles dans le resume (personnes, lieux, montants)
2. **Verification** : Comparaison de chaque entite avec le texte source de reference
3. **Correction** : Suppression ou modification des entites non-fondees

Dans l'exemple ci-dessus, les entites problematiques detectees sont :
- "Milan" (non mentionne dans le texte source, qui indique "Geneve")
- "200k euros" (le texte source indique "100k euros")

Cette technique est essentielle pour reduire les hallucinations des LLMs dans les systèmes de production.

### Multi-Result

OpenAI (ou Azure) peut renvoyer plusieurs complétions pour un même prompt.  
On paramètre `number_of_responses=3` dans les settings.  
Ensuite, `get_text_contents(...)` ou `get_chat_message_contents(...)` renvoie un *tableau* de résultats.

In [9]:
# ============================
# Cellule : Multi-Result
# ============================

from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings

settings = OpenAIChatPromptExecutionSettings(
    service_id=service_id,
    # temperature not set: gpt-5-mini only supports default value (1)
    number_of_responses=3
)

# Utiliser le service chat existant (pas de modele legacy)
chat_service = kernel.get_service(service_id)
chat_history_multi = ChatHistory()
chat_history_multi.add_user_message("Donne-moi une brève blague sur les chats :")

# get_chat_message_contents() => liste de reponses
responses = await chat_service.get_chat_message_contents(chat_history_multi, settings=settings)

for i, r in enumerate(responses):
    print(f"Réponse n°{i+1}:\n{r}\n")

Réponse n°1:
Pourquoi les chats aiment-ils les ordinateurs ? Pour attraper la souris.

Réponse n°2:
Pourquoi le chat aime l'ordinateur ? Parce qu'il adore la souris !

Réponse n°3:
Pourquoi les chats adorent-ils les ordinateurs ? Parce qu'il y a une souris.



### Interprétation : Multi-Result Generation

**Sortie obtenue** : Trois blagues différentes générées en un seul appel API avec `number_of_responses=3`.

| Axe | Ce que la sortie montre | Ce qu'on en retient |
|-----|--------------------------|----------------------|
| **Nombre** | `number_of_responses=3` → trois textes distincts en un seul appel | La cellule ci-dessus imprime les trois ; ils changent à chaque exécution |
| **Genre** | Les trois réponses reposent sur le même jeu de mots (« souris » = animal / périphérique) | Le prompt est identique : la variabilité porte sur la formulation, pas sur l'angle |
| **Conséquence** | Trois variantes, pas trois angles différents | Pour obtenir de la diversité, il faut varier le prompt ou la température, pas le nombre de réponses |

**Points clés** :

1. **Avantages du Multi-Result** :
   - **Efficacité** : Un seul appel API au lieu de 3 appels séquentiels
   - **Coût** : Réduit les coûts de latence réseau et overhead API
   - **Diversité** : Variations générées en parallèle avec la même température
   - **UX** : Permet à l'utilisateur de choisir la meilleure réponse

2. **Paramètres de configuration** :
   ```python
   settings = OpenAIChatPromptExecutionSettings(
       service_id=service_id,
       # temperature non définie : gpt-5-mini ne supporte que la valeur par défaut (1)
       number_of_responses=3    # Nombre de variantes
   )
   ```

3. **Cas d'usage typiques** :

   | Scénario | Nombre de réponses | Post-traitement |
   |----------|-------------------|-----------------|
   | **Brainstorming** | 5-10 | Sélection humaine |
   | **A/B testing** | 2-3 | Métriques engagement |
   | **Génération créative** | 3-5 | Vote utilisateur |
   | **Robustesse** | 3 | Consensus majoritaire |

4. **Considérations de coût** :
   ```
   Coût = nombre_réponses × tokens_par_réponse × prix_par_token
   
   Exemple (GPT-3.5-turbo) :
   - 1 réponse : 60 tokens × $0.002/1K = $0.00012
   - 3 réponses : 180 tokens × $0.002/1K = $0.00036
   ```

**Note technique** : `get_text_contents()` retourne une liste, contrairement à `get_text_content()` (singulier) qui retourne un seul résultat. L'API ChatCompletion a un équivalent avec `get_chat_message_contents()`.

**Comparaison avec approches alternatives** :
```python
# ❌ Approche naïve (3 appels séquentiels)
for i in range(3):
    response = await service.get_text_content(prompt, settings)
# Problème : 3× latence réseau, 3× overhead API

# ✅ Multi-Result (1 appel parallèle)
responses = await service.get_text_contents(prompt, settings)
# Avantage : Latence minimale, coût optimisé
```

**Cas d'usage avancés** :
- **Génération de variantes marketing** : Plusieurs versions d'un slogan
- **Traduction avec nuances** : Différentes formulations d'une traduction
- **Résumé multi-angle** : Résumés avec différents niveaux de détail
- **Code generation** : Plusieurs implémentations d'un algorithme

### Exercice 3 : Analyseur de diversite des reponses multiples

**Objectif** : Implementer une fonction qui analyse la diversite lexicale entre les reponses generees par `number_of_responses=N` et selectionne la meilleure selon un critere.

Quand on genere plusieurs reponses, il est utile de mesurer leur diversite (pour eviter des reponses trop similaires) et de sélectionner automatiquement la meilleure (la plus longue, la plus diverse, etc.).

**Indices** :
- `# Étape 1` : Extraire les mots uniques de chaque reponse avec `set(response.split())`
- `# Étape 2` : Calculer le coefficient de Jaccard entre chaque paire de reponses (intersection / union)
- `# Étape 3` : Sélectionner la reponse avec le plus grand vocabulaire unique
- `# Indice` : `len(set1 & set2) / len(set1 | set2)` donne le score de Jaccard

In [10]:
def analyze_diversity(responses: list[str]) -> dict:
    """
    Analyse la diversite lexicale entre plusieurs reponses.
    
    # Etape 1 : Convertir chaque reponse en ensemble de mots uniques
    # Etape 2 : Calculer les scores de Jaccard entre chaque paire de reponses
    # Etape 3 : Identifier la reponse la plus riche (plus grand vocabulaire unique)
    
    Returns:
        dict avec avg_jaccard, best_response_idx, diversite_scores
    """
    # TODO etudiant : implementer l'analyse de diversite
    return {"avg_jaccard": 0.0, "best_response_idx": 0, "diversite_scores": []}

# Test avec les reponses generees precedemment
# Les reponses du Multi-Result ci-dessus (variable responses) servent de corpus reel
test_responses = [str(r) for r in responses]

result = analyze_diversity(test_responses)
print(f"Score Jaccard moyen : {result['avg_jaccard']:.3f}")
print(f"Reponse la plus riche : #{result['best_response_idx'] + 1}")
for i, score in enumerate(result.get('diversite_scores', [])):
    print(f"  Reponse {i+1} : richesse = {score:.2f}")

Score Jaccard moyen : 0.000
Reponse la plus riche : #1


## Conclusion et Resume

Dans ce notebook, nous avons explore les fonctionnalites avancees de Semantic Kernel :

| Concept | Description | Statut |
|---------|-------------|--------|
| **Chat avec KernelArguments** | Gestion de l'historique et du contexte | Fondamental |
| **Function Calling** | Orchestration automatique via `FunctionChoiceBehavior.Auto()` | **Moderne** (remplace Planners) |
| **Memoire & Embeddings** | Stockage vectoriel avec `InMemoryStore` | **API moderne** |
| **Hugging Face** | Integration de modèles open-source | Optionnel |
| **Groundedness Checking** | Reduction des hallucinations | Production |
| **Multi-Result** | Generations multiples en un appel | Creatif |

### Points cles a retenir

1. **Function Calling remplace les Planners** : Plus simple, plus fiable, moins de tokens
2. **API Memory modernisee** : `InMemoryStore` + `OpenAITextEmbedding` (voir notebook 05 pour Qdrant)
3. **Groundedness** : Essentiel pour les systèmes de production

### Prochaines étapes

- **[03-Agents](03-SemanticKernel-Agents.ipynb)** : Agents avec ChatCompletionAgent, AgentGroupChat
- **[04-Filters](04-SemanticKernel-Filters-Observability.ipynb)** : Intercepter et logger les appels
- **[05-VectorStores](05-SemanticKernel-VectorStores.ipynb)** : RAG complet avec Qdrant

***

**Navigation** : [<< 01-Intro](01-SemanticKernel-Intro.ipynb) | [Index](README.md) | [03-Agents >>](03-SemanticKernel-Agents.ipynb)